# Module-Level Depression Subtyping Analysis

## Overview

Implements a comprehensive pipeline for identifying depression subtypes based on module-level brain connectivity patterns. The analysis uses module internal and external functional connectivity (FC), structural connectivity (SC), and structure-function coupling (SFC) to discover patient subgroups.

## What This Notebook Does

### 1. **Data Processing**
- Computes module-level connectivity metrics from subject-level functional (FC) and structural (SC) connectivity matrices
- Calculates structure-function coupling (SFC) for each brain module/community
- Processes both **internal** (within-module) and **external** (between-module) connectivity patterns

### 2. **Hierarchical Clustering**
- Identifies depression subtypes (clusters) based on module connectivity profiles using Ward's linkage method
- Clustering is performed separately for each:
  - Connectivity type: functional, structural, SFC
  - Direction: internal vs. external module connectivity
- Uses bootstrap-based stability analysis to ensure robust cluster assignments

### 3. **Cluster Validation**
- Bootstrap stability analysis (500 iterations)
- Silhouette scoring
- Calinski-Harabasz scoring
- Cross-modality agreement quantification

### 4. **Statistical Inference**
- Quantile regression models (via rpy2/R) comparing:
  - Depression vs. Control
  - Cluster 0 vs. Control
  - Cluster 1 vs. Control
  - Cluster 0 vs. Cluster 1
- Multiple testing correction (FDR-BH) applied to all p-values

### 5. **Visualization**
- Covariate distributions (age, sex, motion, comorbidities) across groups/clusters
- Module violin plots with significance annotations
- Brain maps (NIfTI overlays) showing cluster-specific module profiles
- Correlation heatmaps and validation metric curves

## Inputs

### Required Files
1. **Combined cohort CSV** (`combined_cohort_F32.csv`)
   - Must include `eid` (subject ID) and `depression_status` (0=control, 1=depressed)
   - Contains demographic and clinical covariates

2. **Per-subject connectivity matrices**
   - Functional: `{eid}/i2/{eid}_functional_connectivity_matrix_reordered.npy`
   - Structural: `{eid}/i2/{eid}_structural_connectivity_matrix_reordered.npy`
   - Located in cohort directories (F32 and control)

3. **Community labels** (`CM.txt`)
   - Text file mapping atlas ROIs → module IDs
   - Defines the modular structure used for aggregation

4. **Atlas parcellation** (`Schaefer1000_TianS4_combined.nii.gz`)
   - NIfTI file used for brain visualizations

### Key Parameters
- **Connectivity types**: functional, structural, SFC
- **Direction types**: internal (within-module), external (between-module)
- **Number of clusters**: 2 
- **Bootstrap iterations**: 500 for stability analysis
- **Covariates**: Age, sex, motion (fMRI & dMRI), ICD-10 comorbidities (I10, Z864, F419)

## Outputs

### CSV Files
1. **`module_connectivity_wide.csv`**
   - Wide-format table: subjects × module features
   - Columns: `{module}_internal_functional`, `{module}_external_structural`, etc.

2. **`module_connectivity_features_with_clusters.csv`**
   - Same as above + cluster assignments for each modality
   - New columns: `functional_internal_cluster`, `sfc_external_cluster`, etc.

3. **`module_connectivity_features_with_covariates.csv`**
   - Final combined table with features, clusters, and covariates
   - Used for all downstream statistical analyses

4. **`module_connectivity_clustering_validation.csv`**
   - Validation metrics (silhouette, Calinski-Harabasz) for k=2-10

5. **Per-subject module connectivity CSVs**
   - `{eid}/i2/functional_module_connectivity.csv`
   - `{eid}/i2/structural_module_connectivity.csv`
   - `{eid}/i2/module_sfc.csv`

### Figures
- **Distribution plots**: Module-level connectivity histograms by cohort
- **Correlation matrices**: Spearman correlations with FDR-corrected significance
- **Dendrograms**: Hierarchical clustering trees for each modality
- **Violin plots**: Module connectivity by group/cluster with statistical annotations
- **Brain maps**: NIfTI overlays showing cluster differences
- **Heatmaps**: Cluster × module feature profiles

### Warning Logs
- **`sfc_warning_logs/sfc_internal_warnings.csv`**
- **`sfc_warning_logs/sfc_external_warnings.csv`**
- Track ConstantInputWarning occurrences during SFC computation
- Include per-module NaN counts and fractions

## Dependencies

### Python Packages
- **Data/Stats**: NumPy, pandas, scipy, scikit-learn, statsmodels
- **Visualization**: matplotlib, seaborn
- **Neuroimaging**: nibabel, nilearn
- **R Interface**: rpy2 (requires R with `quantreg` and `multcomp` packages)

### R Packages (accessed via rpy2)
- `quantreg`: Quantile regression
- `multcomp`: Multiple comparisons

## Analysis Architecture

The pipeline follows a sequential workflow:

```
Raw Connectivity Matrices (ROI × ROI)
          ↓
Module Aggregation (using community labels)
          ↓
Per-Subject Module Features (FC, SC, SFC)
          ↓
Wide DataFrame (subjects × module features)
          ↓
Hierarchical Clustering (Ward's method)
          ↓
Bootstrap Stability & Validation
          ↓
Statistical Testing (Quantile Regression + FDR)
          ↓
Visualization & Reporting
```

## Notes

- **Module labels** must be provided from prior network partitioning (e.g., Louvain, Leiden, or anatomical atlas)
- **Clustering** operates only on depressed subjects; controls are included for statistical comparisons
- **Motion metrics** are modality-specific:
  - `p24441_i2`: rfMRI motion (for FC and SFC)
  - `p24453_i2`: dMRI motion (for SC and SFC)
- All **p-values** undergo FDR-BH correction to control false discovery rate



## Setup and Configuration

In [ ]:
import os
import sys
import numpy as np
import pandas as pd

# Add source code to path
sys.path.append('../source_code')

from clusters.module_clustering_utils import (
    setup_r_environment,
    load_combined_cohort_data,
    get_subject_ids_by_status,
    compute_and_save_module_connectivity,
    build_module_connectivity_dataframe,
    plot_module_metric_distributions,
    plot_module_correlation_matrices,
    perform_module_hierarchical_clustering,
    compute_clustering_validation,
    plot_clustering_validation_metrics,
    analyze_cross_modality_agreement,
    merge_covariates_for_module,
    determine_covariate_distributions,
    run_module_quantile_regression_pipeline,
    plot_cluster_feature_brainmaps,
)

print("✓ Imported module clustering utilities successfully")

In [ ]:
# ==============================================================================
# CONFIGURATION PARAMETERS
# ==============================================================================

# Analysis parameters
CONNECTIVITY_TYPES = ('functional', 'structural', 'sfc')
DIRECTION_TYPES = ('internal', 'external')
ICD_10_COVARIATES = ['I10', 'Z864', 'F419']  # Hypertension, personal history of disease, unspecified anxiety
PROGRESS_EVERY = 50  # Print progress every N subjects
CLUSTER_STABILITY_BOOTSTRAP_ITER = 500

# Directory paths
DEPRESSION_DIR = '.../F32_notask_STRCO_RSSCHA_RSTIA'
CONTROL_DIR = '.../control_notask_STRCO_RSSCHA_RSTIA'
GENERAL_DIR = '.../UKB/cohorts'
COMBINED_COHORT_PATH = '.../UKB/cohorts/combined_cohort_F32.csv'

PLOTS_DIR = '.../schaefer1000+tian54'
FIGURES_DIR = '.../schaefer1000+tian54'

# Motion metrics (modality-specific)
fMRI_MOTION_METRIC = 'p24441_i2'  # rfMRI motion
dMRI_MOTION_METRIC = 'p24453_i2'  # dMRI motion

# Bootstrap iterations for quantile regression (R) standard errors and confidence intervals
QUANTILE_REGRESSION_BOOTSTRAP_R = 5000
# Bootstrap iterations for clustering stability analysis
CLUSTER_STABILITY_BOOTSTRAP_ITER = 5000

# Atlas and community labels
COMMUNITY_LABELS_PATH = '.../notebooks/CM.txt'
ATLAS_FILE = '.../Schaefer1000_TianS4_combined.nii.gz'

# Create output directories if they don't exist
os.makedirs(PLOTS_DIR, exist_ok=True)
os.makedirs(FIGURES_DIR, exist_ok=True)

print("✓ Configuration complete")
print(f"  - Connectivity types: {CONNECTIVITY_TYPES}")
print(f"  - Direction types: {DIRECTION_TYPES}")
print(f"  - Bootstrap iterations: {CLUSTER_STABILITY_BOOTSTRAP_ITER}")

## Step 1: Initialize R Environment
The analysis requires R for quantile regression (`quantreg` package). This step initializes the `rpy2` interface and checks for necessary R packages.

In [ ]:
print("\n[1/9] Initializing R environment...")
setup_r_environment()
print("R packages loaded: quantreg, base, utils, stats")

## Step 2: Load Cohort Data

Load the combined cohort CSV containing subject IDs, depression status, and covariates. We'll separate subjects into depressed (F32) and control groups.

In [ ]:
print("[STEP 2/9] Loading cohort data...")

# Load combined cohort data
data = load_combined_cohort_data(COMBINED_COHORT_PATH)

# Get subject IDs by depression status
depressed_subject_ids, control_subject_ids = get_subject_ids_by_status(data)

print(f"✓ Found {len(depressed_subject_ids)} depressed subjects (cohort F32)")
print(f"✓ Found {len(control_subject_ids)} control subjects")
print(f"\nCohort data shape: {data.shape}")
print(f"Columns: {list(data.columns[:10])}...")

## Step 3: Compute Module Connectivity Features

For each subject, we compute:
- **Functional connectivity (FC)**: Pearson correlation between ROI timeseries
- **Structural connectivity (SC)**: Fiber tract counts/densities between ROIs
- **Structure-Function Coupling (SFC)**: Correlation between FC and SC patterns

Each metric is calculated at the module level for:
- **Internal connectivity**: Within-module connections
- **External connectivity**: Between-module connections

This step processes all subjects and saves per-subject CSV files.

**Note**: This is the most computationally intensive step. Progress is printed every 50 subjects.

In [ ]:
print("[STEP 3/9] Computing module connectivity and coupling features...")

# Load community labels (module assignments for each ROI)
CM = np.loadtxt(COMMUNITY_LABELS_PATH)
print(f"✓ Loaded community labels: {len(CM)} ROIs, {len(np.unique(CM))} modules")

In [ ]:
# Process depressed cohort (F32)
print("\nProcessing cohort F32...")
compute_and_save_module_connectivity(
    DEPRESSION_DIR,
    depressed_subject_ids,
    CM,
    progress_every=PROGRESS_EVERY,
    sfc_warning_log_dir=os.path.join(GENERAL_DIR, 'sfc_warning_logs'),
    cohort_label='F32',
)
print("✓ F32 cohort processing complete")

In [ ]:
# Process control cohort
print("\nProcessing control cohort...")
compute_and_save_module_connectivity(
    CONTROL_DIR,
    control_subject_ids,
    CM,
    progress_every=PROGRESS_EVERY,
    sfc_warning_log_dir=os.path.join(GENERAL_DIR, 'sfc_warning_logs'),
    cohort_label='control',
)
print("✓ Control cohort processing complete")

## Step 4: Build Module Connectivity DataFrame

Aggregate per-subject module connectivity CSVs into a single wide-format DataFrame where:
- **Rows**: Subjects
- **Columns**: Module features (e.g., `1_internal_functional`, `1_external_structural`, `1_internal_sfc`, ...)

This creates the primary dataset for clustering and statistical analysis.

In [ ]:
print("[STEP 4/9] Building module connectivity dataframe...")

# Build wide-format DataFrame
final_df, mods = build_module_connectivity_dataframe(
    DEPRESSION_DIR,
    CONTROL_DIR,
    depressed_subject_ids,
    control_subject_ids,
)

# Save to CSV
final_df.to_csv(os.path.join(GENERAL_DIR, 'module_connectivity_wide.csv'), index=False)

print(f"✓ Combined module connectivity data shape: {final_df.shape}")
print(f"✓ Number of modules: {len(mods)}")
print(f"✓ Modules: {mods}")
print(f"\nFirst few columns: {list(final_df.columns[:10])}")

# Display sample of the data
final_df.head()

## Step 5: Plot Module Metric Distributions And Module Correlation Matrices

Visualize the distribution of module connectivity metrics across cohorts:
- Histograms for each module × connectivity type × direction
- Separate distributions for depressed vs. control subjects

In [ ]:
print("[STEP 5/9] Plotting module metric distributions...")

plot_module_metric_distributions(final_df, mods, PLOTS_DIR)

print("✓ Module distribution plots saved to:", PLOTS_DIR)

Generate Spearman correlation matrices showing relationships between module features:
- Within each connectivity type and direction
- FDR-corrected significance markers

In [ ]:
print("[STEP 5/9] Plotting module correlation matrices...")

plot_module_correlation_matrices(final_df, PLOTS_DIR)

print("✓ Correlation matrix plots saved to:", PLOTS_DIR)

## Step 6: Hierarchical Clustering and Bootstrap Stability

Perform hierarchical clustering on depressed subjects to identify subtypes:

### Clustering Approach
- **Method**: Ward's linkage (minimizes within-cluster variance)
- **Number of clusters**: 2 (can be adjusted)
- **Feature space**: Module connectivity profiles (separately for each modality)

### Bootstrap Stability
- Resample subjects with replacement 500 times
- Re-cluster each bootstrap sample
- Compute per-subject stability (proportion of times assigned to same cluster)
- Calculate Jaccard similarity and NMI for each bootstrap iteration

### Outputs
- Dendrograms showing hierarchical structure
- Cluster assignments added to `final_df`
- Bootstrap stability metrics saved to plots

In [ ]:
print("[STEP 6/9] Hierarchical clustering and bootstrap stability...")

# Perform clustering for each connectivity type × direction
final_df = perform_module_hierarchical_clustering(
    final_df,
    PLOTS_DIR,
    FIGURES_DIR,
    conn_types=CONNECTIVITY_TYPES,
    dir_types=DIRECTION_TYPES,
    n_clusters=2,
    bootstrap_iter=CLUSTER_STABILITY_BOOTSTRAP_ITER,
)

# Save DataFrame with cluster assignments
final_df.to_csv(
    os.path.join(GENERAL_DIR, 'module_connectivity_features_with_clusters.csv'),
    index=False,
)

print("✓ Clustering complete for all modalities")
print("✓ Cluster assignments saved to CSV")

# Show cluster columns
cluster_cols = [col for col in final_df.columns if 'cluster' in col]
print(f"\nCluster assignment columns: {cluster_cols}")

# Show cluster distributions
print("\nCluster distributions (depressed subjects only):")
depressed_df = final_df[final_df['depression_status'] == 1]
for col in cluster_cols:
    if col in depressed_df.columns:
        print(f"  {col}: {depressed_df[col].value_counts().to_dict()}")

## Step 7: Clustering Validation

Evaluate clustering quality using multiple metrics:

### Silhouette Score
- Measures how similar a subject is to its own cluster vs. other clusters
- Range: [-1, 1], higher is better

### Calinski-Harabasz Index
- Ratio of between-cluster to within-cluster variance
- Higher values indicate better-defined clusters

### Cross-Modality Agreement
- Compares cluster assignments across different connectivity types
- Helps identify consistent vs. modality-specific subtypes

Validation is performed for k=2 to k=20 clusters to assess optimal cluster number.

In [ ]:
print("[STEP 7/9] Clustering validation...")

# Compute validation metrics
validation_df, summary = compute_clustering_validation(
    final_df,
    conn_types=CONNECTIVITY_TYPES,
    dir_types=DIRECTION_TYPES,
)

# Save validation results
validation_df.to_csv(
    os.path.join(GENERAL_DIR, 'module_connectivity_clustering_validation.csv'),
    index=False,
)

# Print summary
print("\n" + "="*80)
print("CLUSTERING VALIDATION SUMMARY")
print("="*80)
for row in summary:
    print(
        f"{row['connectivity_type']:12s} {row['direction_type']:8s} | "
        f"Best k={row['best_k_silhouette']} | "
        f"Silhouette={row['best_silhouette_score']:.4f} | "
        f"Calinski-Harabasz={row['best_calinski_harabasz_score']:.2f}"
    )
print("="*80)

# Plot validation curves
plot_clustering_validation_metrics(validation_df, PLOTS_DIR)
print("\n✓ Validation plots saved to:", PLOTS_DIR)

## Step 8: Cross-Modality Agreement Analysis

Analyze how well cluster assignments agree across different connectivity modalities:

### Pairwise Agreement
- Proportion of subjects assigned to the same cluster across modality pairs
- E.g., functional_internal vs. structural_external

### Visualization
- Agreement heatmap
- Cluster distribution bar plots
- Multi-panel overlays showing subjects with identical labels

In [ ]:
print("[STEP 9/9] Cross-modality agreement analysis...")

analyze_cross_modality_agreement(GENERAL_DIR, PLOTS_DIR)

print("✓ Cross-modality agreement plots saved to:", PLOTS_DIR)

## Step 9: Covariates, Statistical Testing, and Visualization

### 9.1 Merge Covariates With Connectivity Dataframe and Plot Distributions

The `plot_covariate_distributions` function analyzes covariate associations across groups and clusters.

**Outputs**:
- **PNG files**: Visualization plots showing covariate distributions
- **CSV files**: Detailed statistical results including group comparisons and effect sizes
- **1 TXT file**: Comprehensive summary containing descriptive statistics and inference testing results (chi-square tests, t-tests, ANOVA, etc. as appropriate for each covariate type)

In [ ]:
print("[STEP 9/9] Covariates, quantile regression, and cluster profiles...")

# Define covariate columns to merge
covariate_columns = [
    'eid',           # Subject ID
    'p21003_i2',     # Age
    'p31',           # Sex
    'I10',           # Hypertension
    'Z864',          # Personal history of disease
    'F419',          # Unspecified anxiety disorder
    'p24453_i2',     # dMRI motion
    'p24441_i2',     # fMRI motion
]

# Merge covariates with module features
merged, colname_map = merge_covariates_for_module(
    final_df,
    data,
    covariate_columns,
    GENERAL_DIR,
    output_basename='module_connectivity_features_with_covariates',
)

print(f"✓ Merged covariates. Final dataset shape: {merged.shape}")
print(f"✓ Saved to: {os.path.join(GENERAL_DIR, 'module_connectivity_features_with_covariates.csv')}")

In [ ]:
# Plot covariate distributions across groups and clusters
# This function outputs PNG files, CSV files with detailed stats, and 1 TXT file
# with descriptive statistics and inference testing summaries
print("\nPlotting covariate distributions...")

determine_covariate_distributions(
    combined_df=merged,
    available_types=[],
    motion_metric=fMRI_MOTION_METRIC,
    out_dir=PLOTS_DIR,
    cohorts_dir=None,
    icd_covariates=ICD_10_COVARIATES,
    conn_types=list(CONNECTIVITY_TYPES),
    dir_types=list(DIRECTION_TYPES),
)

print("✓ Covariate plots (PNG), statistical CSVs, and summary TXT file saved to:", PLOTS_DIR)

### 9.2 Quantile Regression Analysis

The `run_module_quantile_regression_pipeline` function conducts quantile regression analyses to assess median differences for depression vs control, clusters vs control and clusters vs each other for each module feature (so univariate fashion), adjusting for covariates. Uses bootstrap resampling to estimate confidence intervals and applies FDR correction for multiple comparisons. Also uses bootstrapping to estimate standard errors. Additionally, generates visualizations of regression results (violin plots with significance annotations), saves brain modules with significant between-cluster differences for subsequent brainmaps, saves FDR-correction log and overall R output log.

**Outputs**:
- **PNG files**: Visualization plots showing regression results
- **TXT files**: FDR-correction log and overall R output log

In [ ]:
# Run quantile regression pipeline
print("\nRunning quantile regression pipeline...")
print("  This may take several minutes...")

brainmap_significance = run_module_quantile_regression_pipeline(
    merged=merged,
    final_df=final_df,
    mods=mods,
    R=QUANTILE_REGRESSION_BOOTSTRAP_R,
    conn_types=CONNECTIVITY_TYPES,
    dir_types=DIRECTION_TYPES,
    icd_covariates=ICD_10_COVARIATES,
    fMRI_MOTION_METRIC=fMRI_MOTION_METRIC,
    dMRI_MOTION_METRIC=dMRI_MOTION_METRIC,
    plots_dir=PLOTS_DIR,
    depressed_subjects_dir=DEPRESSION_DIR,
    colname_map=colname_map,
)

print("\n✓ Quantile regression complete")
print("✓ Violin plots with significance annotations saved to:", PLOTS_DIR)

### 9.3 Brain Map Visualization of Cluster Differences
The function `plot_cluster_feature_brainmaps` generates brain map visualizations for modules with significant between-cluster differences. It creates Nilearn plots and Nifti masks of cluster median profiles and of cluster median differences for significant modules. Nifti masks can be used for subsequent brainmap visualizations in MRIcroGL.

In [ ]:
# Generate brain maps 
print("\nGenerating brain maps...")

# Prepare cluster data (depressed subjects only)
final_df_clusters = final_df.loc[final_df['depression_status'] == 1].reset_index(drop=True)
cluster_cols = [
    f'{conn}_{dir_type}_cluster'
    for conn in CONNECTIVITY_TYPES
    for dir_type in DIRECTION_TYPES
]
feature_df = final_df_clusters.drop(
    columns=[c for c in ['eid', 'depression_status'] + cluster_cols if c in final_df_clusters.columns]
).copy()

# Generate brain maps
plot_cluster_feature_brainmaps(
    final_df=final_df_clusters,
    feature_df=feature_df.assign(eid=final_df_clusters['eid']),
    atlas_img_path=ATLAS_FILE,
    community_labels=CM,
    fig_dir=FIGURES_DIR,
    modules=mods,
    minmax=True,
    significance_map=brainmap_significance,
    pvalue_threshold=0.05,
)

print("✓ Brain maps saved to:", FIGURES_DIR)

## Analysis Complete!

### Summary of Outputs

**CSV Files** (in `data/UKB/cohorts/`):
- `module_connectivity_wide.csv`: Wide-format module features
- `module_connectivity_features_with_clusters.csv`: Features + cluster assignments
- `module_connectivity_features_with_covariates.csv`: Final combined dataset
- `module_connectivity_clustering_validation.csv`: Validation metrics

**Plots** (in `reports/plots/schaefer1000+tian54/`):
- Distribution plots
- Correlation matrices
- Dendrograms
- Validation curves
- Covariate distributions
- Violin plots with significance

**Figures** (in `reports/figures/schaefer1000+tian54/`):
- Brain maps showing cluster differences
- Optional NIfTI masks for significant modules

**Logs** (in `data/UKB/cohorts/sfc_warning_logs/`):
- `sfc_internal_warnings.csv`
- `sfc_external_warnings.csv`
